# Bayesian Thinking

Most machine learning methods produce a point estimate: a single number, a single prediction, a single set of weights. Bayesian methods do something richer — they maintain a distribution over unknowns and update it as data arrives. This notebook covers the mechanics of that update, how it manifests in classifiers and regularizers you already use, and how it powers modern hyperparameter search.

## Probabilistic Reasoning and Bayes' Rule

Let $\theta$ be an unknown quantity — a model parameter, a coin bias, a disease prevalence. Before seeing any data we encode our uncertainty as a **prior distribution** $p(\theta).$ After observing data $\mathcal{D}$ we form the **posterior distribution** via Bayes' rule:

$$
\boxed{
p(\theta \mid \mathcal{D}) = \frac{p(\mathcal{D} \mid \theta)\, p(\theta)}{p(\mathcal{D})} \propto p(\mathcal{D} \mid \theta)\, p(\theta).
}
$$

The three factors have names: $p(\mathcal{D} \mid \theta)$ is the **likelihood** (how probable is the data under a fixed $\theta$); $p(\theta)$ is the **prior**; and $p(\theta \mid \mathcal{D})$ is the **posterior**. The denominator $p(\mathcal{D}) = \int p(\mathcal{D} \mid \theta) p(\theta)\, d\theta$ is a normalizing constant that does not depend on $\theta$ and is often intractable to compute — but for prediction and point estimation we only need the unnormalized form on the right.

**Bayesian update.** Bayes' rule is applied sequentially: the posterior after observing $\mathcal{D}_1$ becomes the prior for updating on $\mathcal{D}_2$. Formally, $p(\theta \mid \mathcal{D}_1, \mathcal{D}_2) \propto p(\mathcal{D}_2 \mid \theta) \cdot p(\theta \mid \mathcal{D}_1).$ This means the order of evidence does not matter — only the total data does.

**Conjugate priors.** In general, computing the posterior requires numerical methods. A **conjugate prior** is a prior distribution that, when combined with the corresponding likelihood, yields a posterior in the same family. Conjugacy makes inference closed-form and analytically elegant.

**Beta-Binomial.** The canonical example is estimating a coin bias $p \in [0, 1].$ We model $k$ heads in $n$ flips as Binomial$(n, p)$ and place a Beta$(\alpha, \beta)$ prior on $p$. The posterior is:

$$
p \mid k, n \;\sim\; \text{Beta}(\alpha + k,\; \beta + n - k).
$$

The hyperparameters $\alpha$ and $\beta$ can be interpreted as pseudocounts of heads and tails seen before any real data. A flat prior $\text{Beta}(1, 1)$ corresponds to no prior knowledge.

**Normal-Normal.** For a Gaussian likelihood with known variance $\sigma^2$, a Gaussian prior on the mean $\mu \sim \mathcal{N}(\mu_0, \tau^2)$ is conjugate. The posterior mean is a precision-weighted average of the prior mean and the sample mean:

$$
\mu \mid \mathbf{x} \;\sim\; \mathcal{N}\!\left(\frac{\tau^2}{\tau^2 + \sigma^2/n}\bar{x} + \frac{\sigma^2/n}{\tau^2 + \sigma^2/n}\mu_0,\;\left(\frac{1}{\tau^2} + \frac{n}{\sigma^2}\right)^{-1}\right).
$$

As $n \to \infty$ the posterior concentrates around the sample mean and the prior becomes irrelevant — a general property of consistent Bayesian estimators.

**Posterior predictive distribution.** After updating the posterior, we make predictions by averaging over all parameter values weighted by their posterior probability:

$$
p(x_{\text{new}} \mid \mathcal{D}) = \int p(x_{\text{new}} \mid \theta)\, p(\theta \mid \mathcal{D})\, d\theta.
$$

This marginalizes out $\theta$ rather than plugging in a point estimate. The posterior predictive is always broader than the predictive distribution under the MAP estimate, because it accounts for remaining uncertainty in $\theta$ itself.

Sequential Bayesian updating with the Beta-Binomial model, visualized after $n \in \{0, 5, 20, 100\}$ coin flips from a true bias $p^* = 0.7$:

In [ ]:
#| label: fig-prior-posterior
#| fig-cap: "Sequential Bayesian updating with the Beta-Binomial model. Each panel shows the posterior after observing $n$ coin flips (true $p^* = 0.7$). The posterior concentrates around the true probability as data accumulates."
#| code-fold: true

import numpy as np
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline
from scipy.stats import beta as beta_dist

backend_inline.set_matplotlib_formats("svg")

TRUE_P = 0.7
ALPHA0, BETA0 = 1, 1      # flat prior Beta(1,1)
rng = np.random.default_rng(42)

ns = [0, 5, 20, 100]
flips = rng.binomial(1, TRUE_P, size=max(ns))

fig, axes = plt.subplots(1, 4, figsize=(12, 3), sharey=False)

theta = np.linspace(0, 1, 500)

for ax, n in zip(axes, ns):
    k = int(flips[:n].sum()) if n > 0 else 0   # <1>
    alpha_post = ALPHA0 + k                     # <2>
    beta_post  = BETA0  + n - k

    pdf = beta_dist.pdf(theta, alpha_post, beta_post)
    ax.plot(theta, pdf, color="C0", linewidth=2)
    ax.fill_between(theta, pdf, alpha=0.15, color="C0")
    ax.axvline(TRUE_P, color="C1", linestyle="dashed", linewidth=1.5,
               label=r"$p^*=0.7$")

    ax.set_title(
        f"$n={n}$,  $k={k}$\n"
        rf"Beta$({alpha_post},{beta_post})$",
        fontsize=9
    )
    ax.set_xlabel(r"$p$")
    ax.set_xlim(0, 1)
    ax.grid(linestyle="dotted", alpha=0.6)
    if ax is axes[0]:
        ax.set_ylabel("density")
        ax.legend(fontsize=8)

fig.tight_layout()
plt.show();

The procedure for each panel is:

1. Accumulate heads in the first $n$ flips.
2. Apply the conjugate update: $\alpha \leftarrow \alpha_0 + k$, $\beta \leftarrow \beta_0 + n - k$.

**Figure.** With $n = 0$ the posterior is the flat prior. After five flips the distribution is wide but already shifted toward $0.7.$ By $n = 100$ it is tightly concentrated — the specific $\alpha, \beta$ values encode everything the data has told us, and the prior influence has effectively vanished.

## Naive Bayes Classifiers

**Naive Bayes** applies Bayes' rule directly to classification. Given a class label $y$ and a feature vector $\mathbf{x} = (x_1, \ldots, x_d)$, the MAP prediction is:

$$
\hat{y} = \operatorname{argmax}_{y} \; p(y) \prod_{j=1}^d p(x_j \mid y),
$$

where the product follows from the **conditional independence assumption**: $p(x_1, \ldots, x_d \mid y) = \prod_j p(x_j \mid y).$ This is the "naive" part — features are assumed independent given the class, which is almost never literally true but is computationally extremely convenient. With $d$ features and $K$ classes, we need only $O(dK)$ parameters instead of $O(K^d)$ for the full joint.

**Gaussian NB.** For continuous features, we model each $p(x_j \mid y)$ as a Gaussian with class-specific mean $\mu_{jy}$ and variance $\sigma_{jy}^2$ estimated from training data. The class-conditional density is:

$$
p(x_j \mid y) = \frac{1}{\sqrt{2\pi\sigma_{jy}^2}} \exp\!\left(-\frac{(x_j - \mu_{jy})^2}{2\sigma_{jy}^2}\right).
$$

**Multinomial NB.** For text classification with bag-of-words counts, we model $p(x_j \mid y)$ as a Multinomial distribution. Smoothing the counts by adding a pseudocount $\alpha > 0$ to every word (Laplace smoothing) prevents zero probabilities for unseen words and is equivalent to placing a Dirichlet prior over the word probabilities.

**Log-probability for numerical stability.** In practice we work in log-space. Multiplying many small probabilities underflows to zero even for moderate $d$. Taking the log converts the product to a sum:

$$
\hat{y} = \operatorname{argmax}_{y} \left[ \log p(y) + \sum_{j=1}^d \log p(x_j \mid y) \right].
$$

**Why it works despite strong assumptions.** The conditional independence assumption is violated in virtually every real dataset. Yet Naive Bayes is often competitive because classification accuracy only requires that the ranking of class posteriors be correct, not that the posterior probabilities be calibrated. Even with severely correlated features the argmax is often unchanged. Additionally, the model has very few parameters relative to its input dimensionality — this bias toward simple structure is actually beneficial in high-dimensional, low-data regimes (text classification being the canonical case).

Comparing Gaussian and Multinomial Naive Bayes on the Iris dataset and a text dataset:

In [ ]:
from sklearn.datasets import load_iris, fetch_20newsgroups
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_score
import numpy as np

# Gaussian NB on Iris
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
gnb_scores = cross_val_score(GaussianNB(), X_iris, y_iris, cv=5)
print(f"Gaussian NB (Iris)     acc = {gnb_scores.mean():.3f} ± {gnb_scores.std():.3f}")

# Multinomial NB on 20 Newsgroups (4 categories)
cats = ["sci.space", "rec.sport.hockey", "comp.graphics", "talk.religion.misc"]
news = fetch_20newsgroups(subset="all", categories=cats, remove=("headers", "footers", "quotes"))
X_text = TfidfVectorizer(max_features=10_000).fit_transform(news.data)  # <1>
mnb_scores = cross_val_score(MultinomialNB(alpha=1.0), X_text, news.target, cv=5)
print(f"Multinomial NB (20news) acc = {mnb_scores.mean():.3f} ± {mnb_scores.std():.3f}")

1. TF-IDF features replace raw counts, which empirically improves Multinomial NB by down-weighting very frequent terms.

## Priors as Regularization

**Maximum a posteriori (MAP) estimation** finds the mode of the posterior rather than fully characterizing it:

$$
\hat{\theta}_{\text{MAP}} = \operatorname{argmax}_{\theta}\; \log p(\mathcal{D} \mid \theta) + \log p(\theta).
$$

The prior $\log p(\theta)$ acts as a regularization term. Crucially, the choice of prior family determines which regularizer appears.

**Gaussian prior = Ridge (L2).** Let $\theta \sim \mathcal{N}(0, \lambda^{-1} \mathbf{I}).$ Then:

$$
\log p(\theta) = -\frac{\lambda}{2} \|\theta\|^2 + \text{const}.
$$

MAP estimation with a Gaussian prior is exactly Ridge regression:

$$
\hat{\theta}_{\text{MAP}} = \operatorname{argmax}_{\theta} \left[ \log p(\mathcal{D} \mid \theta) - \frac{\lambda}{2}\|\theta\|^2 \right].
$$

A Gaussian prior is a soft constraint that pulls weights toward zero, penalizing large weights quadratically. It does not produce sparse solutions because the gradient of $\|\theta\|^2$ is $2\theta$, which only reaches zero exactly at $\theta = 0$.

**Laplace prior = Lasso (L1).** Let $\theta \sim \text{Laplace}(0, \lambda^{-1}).$ Then:

$$
\log p(\theta) = -\lambda \|\theta\|_1 + \text{const}.
$$

MAP estimation with a Laplace prior is exactly Lasso:

$$
\hat{\theta}_{\text{MAP}} = \operatorname{argmax}_{\theta} \left[ \log p(\mathcal{D} \mid \theta) - \lambda\|\theta\|_1 \right].
$$

The Laplace prior has a sharp spike at zero. Because the subgradient of $|\theta_j|$ is the interval $[-1, 1]$ at $\theta_j = 0$, it is genuinely possible for the optimum to lie at exactly $\theta_j = 0$, producing sparse solutions. This is why Lasso performs automatic feature selection while Ridge does not.

**Full Bayes vs. MAP.** MAP is a point estimate — it keeps only the mode of the posterior and discards uncertainty. Full Bayesian inference retains the entire posterior distribution, which matters when (1) the posterior is multimodal or heavily skewed, (2) we need calibrated uncertainty estimates for downstream decisions, or (3) we want to marginalize over parameters in the posterior predictive. For large datasets and smooth posteriors, MAP and the full Bayes posterior mean converge, making MAP a good practical approximation. For small datasets, MAP can be overconfident.

Verifying the MAP = regularized MLE correspondence on a regression task:

In [ ]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
X, y = make_regression(n_samples=200, n_features=20, n_informative=5,
                        noise=10.0, random_state=0)
scaler = StandardScaler()
X = scaler.fit_transform(X)

lam = 1.0                                               # <1>
ridge = Ridge(alpha=lam).fit(X, y)
lasso = Lasso(alpha=lam, max_iter=5000).fit(X, y)

print(f"Ridge  — nonzero weights: {(ridge.coef_ != 0).sum()}/20")
print(f"Lasso  — nonzero weights: {(lasso.coef_ != 0).sum()}/20")
print(f"\nRidge ||w||₂: {np.linalg.norm(ridge.coef_):.3f}")
print(f"Lasso ||w||₁: {np.linalg.norm(lasso.coef_, 1):.3f}")

1. `alpha` in sklearn's Ridge and Lasso corresponds to $\lambda$ in the MAP formulation above — same penalty, different name.

**Result.** Ridge keeps all 20 weights nonzero; Lasso zeros out the uninformative ones. Both behaviors are immediate consequences of the prior geometry.

## Bayesian Optimization

Bayesian optimization (BO) is a strategy for global optimization of expensive black-box functions $f \colon \mathcal{X} \to \mathbb{R}$ where each evaluation of $f$ is costly (e.g. training a model for hours). The core idea: maintain a probabilistic **surrogate model** over $f$, and use it to decide where to evaluate next.

**Gaussian Process surrogate.** A **Gaussian Process** (GP) is a distribution over functions such that any finite collection of function values is jointly Gaussian. A GP is specified by a mean function $m(\mathbf{x})$ and a covariance kernel $k(\mathbf{x}, \mathbf{x}^\prime).$ Given observations $\{(\mathbf{x}_i, y_i)\}_{i=1}^n$ with $y_i = f(\mathbf{x}_i) + \varepsilon_i$, the posterior over $f$ at any new point $\mathbf{x}_*$ is Gaussian with a closed-form mean and variance computable via kernel matrix algebra.

**Acquisition functions.** The GP posterior gives us, at any candidate $\mathbf{x}$, a posterior mean $\mu(\mathbf{x})$ (what we expect $f$ to be) and a posterior standard deviation $\sigma(\mathbf{x})$ (our uncertainty). An **acquisition function** $\alpha(\mathbf{x})$ combines these into a score that trades off exploitation (query where $\mu$ is high) and exploration (query where $\sigma$ is high). Common choices:

**Expected Improvement (EI).** Let $f^+ = \max_i y_i$ be the current best observation. EI is the expected amount by which $f(\mathbf{x})$ exceeds $f^+$:

$$
\text{EI}(\mathbf{x}) = \mathbb{E}_{f(\mathbf{x}) \sim \mathcal{N}(\mu, \sigma^2)}\!\left[ \max(f(\mathbf{x}) - f^+, 0) \right]
= (\mu - f^+)\,\Phi(Z) + \sigma\,\phi(Z),
$$

where $Z = (\mu - f^+) / \sigma$, and $\Phi, \phi$ are the Gaussian CDF and PDF.

**Upper Confidence Bound (UCB).** A simpler alternative: $\text{UCB}(\mathbf{x}) = \mu(\mathbf{x}) + \kappa\,\sigma(\mathbf{x})$ for some exploration parameter $\kappa > 0.$ Larger $\kappa$ increases exploration.

The next query is $\mathbf{x}_{n+1} = \operatorname{argmax}_{\mathbf{x}} \alpha(\mathbf{x}).$ This inner optimization is cheap because evaluating the acquisition function is cheap.

**When to use BO vs. grid/random search.** Grid search scales exponentially with the number of hyperparameters — infeasible beyond 3–4 dimensions. Random search outperforms grid search in high dimensions because most hyperparameters matter little (effective dimensionality is low). BO outperforms both when each evaluation is expensive (say, $> 10$ minutes) and the budget is small (say, $\leq 100$ evaluations). For cheap objectives or very high-dimensional hyperparameter spaces (where the GP surrogate itself becomes expensive), random search with a large budget often wins.

GP surrogate model after five evaluations of a noisy 1D objective, with the Expected Improvement acquisition function:

In [ ]:
#| label: fig-bayesian-optimization
#| fig-cap: "Bayesian optimization after five evaluations of a noisy 1D objective. **(Top)** GP surrogate: mean (blue), ±2 std band, observed points, and true function. **(Bottom)** Expected Improvement acquisition function; the dashed line marks the next query point."
#| code-fold: true

import numpy as np
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel

backend_inline.set_matplotlib_formats("svg")

rng = np.random.default_rng(7)

def f_true(x):
    return np.sin(3 * np.pi * x) + 0.5 * np.cos(6 * np.pi * x)

def expected_improvement(X_cand, gp, y_best, xi=0.01):
    """Vectorised Expected Improvement."""
    mu, sigma = gp.predict(X_cand, return_std=True)
    sigma = np.maximum(sigma, 1e-9)
    Z = (mu - y_best - xi) / sigma
    return (mu - y_best - xi) * norm.cdf(Z) + sigma * norm.pdf(Z)

# Initial observations
X_obs = rng.uniform(0, 1, size=(5, 1))
y_obs = f_true(X_obs.ravel()) + rng.normal(0, 0.1, size=5)  # <1>

# Fit GP
kernel = 1.0 * RBF(length_scale=0.2) + WhiteKernel(noise_level=0.01)
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, random_state=0)
gp.fit(X_obs, y_obs)

# Dense prediction grid
X_grid = np.linspace(0, 1, 500).reshape(-1, 1)
mu, std = gp.predict(X_grid, return_std=True)

# Expected Improvement and next query
ei = expected_improvement(X_grid, gp, y_best=y_obs.max())
x_next = X_grid[ei.argmax(), 0]                              # <2>

# --- Plot ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5),
                                gridspec_kw={"height_ratios": [3, 1]},
                                sharex=True)

# Top panel: GP surrogate
x_plot = X_grid.ravel()
ax1.plot(x_plot, f_true(x_plot), "k--", lw=1.2, label="true $f$", alpha=0.6)
ax1.plot(x_plot, mu, color="C0", lw=2, label="GP mean")
ax1.fill_between(x_plot, mu - 2*std, mu + 2*std,
                 color="C0", alpha=0.15, label=r"$\pm 2\sigma$")
ax1.scatter(X_obs.ravel(), y_obs, color="C1", s=60, zorder=5,
            edgecolors="k", linewidths=0.8, label="observations")
ax1.set_ylabel("$f(x)$")
ax1.legend(fontsize=8, loc="upper right")
ax1.grid(linestyle="dotted", alpha=0.6)

# Bottom panel: EI
ax2.plot(x_plot, ei, color="C2", lw=2)
ax2.axvline(x_next, color="C3", linestyle="dashed", lw=1.5, label=f"next query: $x={x_next:.2f}$")
ax2.fill_between(x_plot, 0, ei, color="C2", alpha=0.15)
ax2.set_xlabel("$x$")
ax2.set_ylabel("EI")
ax2.legend(fontsize=8)
ax2.grid(linestyle="dotted", alpha=0.6)

fig.tight_layout()
plt.show();

Key observations:

1. Observations are noisy: $y_i = f(x_i) + \varepsilon_i$, $\varepsilon_i \sim \mathcal{N}(0, 0.1^2).$ The `WhiteKernel` in the GP accounts for this observation noise.
2. The next query maximizes EI — it is in a region of high uncertainty near a promising area, reflecting the explore-exploit tradeoff.

**Figure.** The GP mean tracks the true function despite only five observations. The uncertainty band is wide wherever no observation exists. The EI acquisition function concentrates mass in a region that is both unexplored and near the current best — a region to the right where the true function has a peak the GP has not yet discovered.

## Hyperparameter Tuning with Optuna

[Optuna](https://optuna.org/) is a hyperparameter optimization framework built around a define-by-run API: you write a plain Python objective function and let Optuna decide what to evaluate. Its default sampler is **TPE** (Tree-structured Parzen Estimator), a model-based search algorithm related to Bayesian optimization.

**TPE.** Rather than fitting a GP over the entire search space, TPE maintains two density models: $\ell(\mathbf{x})$ over configurations that produced good results and $g(\mathbf{x})$ over configurations that did not. The next candidate maximizes the ratio $\ell(\mathbf{x}) / g(\mathbf{x}),$ which is an approximation to Expected Improvement. TPE scales better than GP-based BO for high-dimensional discrete or mixed search spaces — exactly the kind that arise in machine learning.

**Pruning.** Optuna supports early stopping of unpromising trials via **pruners**. During training, the objective function reports an intermediate value via `trial.report(value, step)`, and calls `trial.should_prune()` to check whether Optuna recommends stopping. If so, it raises `optuna.exceptions.TrialPruned`. The `MedianPruner` stops a trial if its intermediate score falls below the median of completed trials at the same step.

Setting up an Optuna study to tune XGBoost on the California housing dataset:

In [ ]:
import optuna
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

optuna.logging.set_verbosity(optuna.logging.WARNING)   # <1>

housing = fetch_california_housing()
X, y = housing.data, housing.target
X = StandardScaler().fit_transform(X)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial):                                   # <2>
    params = {
        "n_estimators":  trial.suggest_int("n_estimators", 50, 500),
        "max_depth":     trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "subsample":     trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "random_state":  0,
        "n_jobs":        -1,
    }
    model = xgb.XGBRegressor(**params, verbosity=0)
    scores = cross_val_score(model, X, y, cv=cv,
                             scoring="neg_root_mean_squared_error")  # <3>
    return scores.mean()                               # <4>

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=0),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5),
)
study.optimize(objective, n_trials=40, show_progress_bar=True)

1. Suppresses per-trial log lines; we only want the summary.
2. The `objective` function takes a `trial` object and returns a scalar. Optuna's TPE sampler proposes parameter values via `trial.suggest_*` calls.
3. `neg_root_mean_squared_error` returns a negative value so that `cross_val_score`'s maximize convention is consistent with Optuna's `direction="maximize"`.
4. We return the mean CV score across folds. Optuna tracks this to model the objective surface.

Reporting the best result:

In [ ]:
print(f"Best CV score (neg-RMSE): {study.best_value:.4f}")
print(f"Best trial: #{study.best_trial.number}")
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k:25s} = {v}")

Visualizing optimization history and parameter importances:

In [ ]:
#| code-fold: true

import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline
import optuna.visualization.matplotlib as optuna_plt

backend_inline.set_matplotlib_formats("svg")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Optimization history
plt.sca(axes[0])                                     # <1>
optuna_plt.plot_optimization_history(study)
axes[0].set_title("Optimization history")
axes[0].grid(linestyle="dotted", alpha=0.6)

# Parameter importances
plt.sca(axes[1])
optuna_plt.plot_param_importances(study)
axes[1].set_title("Parameter importances")
axes[1].grid(linestyle="dotted", alpha=0.6)

fig.tight_layout()
plt.show();

1. `plt.sca` sets the current axes so that `optuna.visualization.matplotlib` renders into the correct panel.

**Result.** The optimization history shows the best value improving steeply in early trials as TPE narrows down the search space, then leveling off as it exploits the most promising region. The parameter importances (computed via functional ANOVA) typically reveal that `learning_rate` and `n_estimators` dominate, while `colsample_bytree` and `subsample` contribute less — useful for deciding which parameters to fix in a follow-up search.

:::{.callout-note}
Optuna's `plot_param_importances` uses the [fANOVA](https://proceedings.mlr.press/v32/hutter14.html) algorithm to attribute variance in the objective to individual hyperparameters, marginalizing over the others. This is more reliable than simply looking at the best trial's parameter values, which can be misleading due to interactions.

:::

---

■